# Zalando Network Planning — Warehouse Inbound Forecast

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.1f}".format)

## 1. Load Data

In [2]:
hist = pd.read_csv(
    "data/202402_Data - Case Study - Analyst Network Planning_vShared - input_1_histrocial.csv",
    parse_dates=["date_order", "date_wh_receive"]
)

fc = pd.read_csv(
    "data/202402_Data - Case Study - Analyst Network Planning_vShared - input_2_sales_fc.csv",
    parse_dates=["date_order"]
)

print("Historical data:", hist.shape)
display(hist.head())

print("\nSales forecast:", fc.shape)
display(fc.head())

Historical data: (1763, 6)


,date_order,day_of_week_order,date_wh_receive,day_of_week_wh_receive,CW,items
0,2022-01-01,6,2022-01-01,6,52,18338
1,2022-01-01,6,2022-01-02,7,52,408
2,2022-01-01,6,2022-01-03,1,1,2551
3,2022-01-01,6,2022-01-04,2,1,11965
4,2022-01-01,6,2022-01-05,3,1,180



Sales forecast: (30, 3)


,date_order,day_of_week_order,Forecated_items
0,2022-06-01,3,80882
1,2022-06-02,4,58562
2,2022-06-03,5,55809
3,2022-06-04,6,48880
4,2022-06-05,7,49208


## 2. Data Quality Checks

### Checking Historical Data

In [3]:
### input_1: Shape and dtypes
print("=== input_1: Historical Data ===")
print(f"Shape: {hist.shape}")
print()
print(hist.dtypes)

=== input_1: Historical Data ===
Shape: (1763, 6)

date_order                datetime64[us]
day_of_week_order                  int64
date_wh_receive           datetime64[us]
day_of_week_wh_receive             int64
CW                                 int64
items                              int64
dtype: object


All data types are loaded in correct format.

In [4]:
### input_1: Missing values
print("=== input_1: Missing Values ===")
print(hist.isnull().sum())

=== input_1: Missing Values ===
date_order                0
day_of_week_order         0
date_wh_receive           0
day_of_week_wh_receive    0
CW                        0
items                     0
dtype: int64


We don't have any nulls in the dataset.

In [5]:
### input_1: Date range
print("=== input_1: Date Range ===")
print(f"date_order:      {hist['date_order'].min().date()} → {hist['date_order'].max().date()}")
print(f"date_wh_receive: {hist['date_wh_receive'].min().date()} → {hist['date_wh_receive'].max().date()}")
print(f"Unique order dates: {hist['date_order'].nunique()}")

=== input_1: Date Range ===
date_order:      2022-01-01 → 2022-05-31
date_wh_receive: 2022-01-01 → 2022-06-11
Unique order dates: 151


Order Dates range from Jan'22 till end of May'22. However, there is a *Spillover from May orders* into June receiving, which needs to be accounted for in June forecast.

In [6]:
### input_1: Negative or zero item counts
neg_hist = hist[hist["items"] <= 0]
print(f"=== input_1: Negative or Zero Items ===")
print(f"Rows with items <= 0: {len(neg_hist)}")
if len(neg_hist) > 0:
    display(neg_hist)

=== input_1: Negative or Zero Items ===
Rows with items <= 0: 0


We don't have any 0 or negative values in 'items' data.

In [7]:
### input_1: Duplicate rows
dupes_hist = hist.duplicated().sum()
print(f"=== input_1: Duplicate Rows ===")
print(f"Duplicate rows: {dupes_hist}")

=== input_1: Duplicate Rows ===
Duplicate rows: 0


No entire duplicate records found.

In [ ]:
### Total items per order date
total_per_order = hist.groupby("date_order")["items"].sum()
total_per_order.head()

date_order
2022-01-01    33614
2022-01-02    52529
2022-01-03    90112
2022-01-04    83831
2022-01-05    66818
Name: items, dtype: int64

In [ ]:
# Calculate share sums per order to whs receiving date
share_sums = (
    hist.groupby(["date_order", "date_wh_receive"])["items"].sum()
    .div(total_per_order, level="date_order")
    
)
share_sums.head()

date_order  date_wh_receive
2022-01-01  2022-01-01        0.5
            2022-01-02        0.0
            2022-01-03        0.1
            2022-01-04        0.4
            2022-01-05        0.0
Name: items, dtype: float64

In [ ]:
# Aggregate share sums by order date to check total shares sum to 1
share_sums = share_sums.groupby("date_order").sum()
share_sums.head()

date_order
2022-01-01   1.0
2022-01-02   1.0
2022-01-03   1.0
2022-01-04   1.0
2022-01-05   1.0
Name: items, dtype: float64

In [22]:
print("=== input_1: Item Share Sum per Order Date ===")
print(share_sums.describe().to_frame("share_sum"))
incomplete = share_sums[share_sums < 0.99]
print(f"\nOrder dates with share sum < 99%: {len(incomplete)}")
if len(incomplete) > 0:
    display(incomplete)

=== input_1: Item Share Sum per Order Date ===
       share_sum
count      151.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0

Order dates with share sum < 99%: 0


Every single order date has a perfect share sum of 1.0 — no missing receipts, no data gaps.

### Data Quality Summary — input_1 (Historical Data)

| Check | Result |
|---|---|
| Shape | 1,763 rows × 6 columns |
| Unique order dates | 151 (Jan 1 – May 31, 2022) |
| Missing values | None |
| Duplicate rows | None |
| Negative / zero items | None |
| Date range | Orders: Jan–May 2022; WH receipts extend to Jun 11, 2022 |
| Share sums per order date | All exactly 1.0 — no data gaps or losses |
| Data types | Dates as `datetime64`, numeric columns as `int64` |

**Overall:** input_1 is clean and complete. Late-May orders spill into early June warehouse receipts and will need to be accounted for in the forecast. No remediation required before modelling.

In [ ]:
### input_2: Shape, dtypes, missing values
print("=== input_2: Sales Forecast ===")
print(f"Shape: {fc.shape}")
print()
print(fc.dtypes)
print()
print("Missing values:")
print(fc.isnull().sum())

In [ ]:
### input_2: Date range and gaps
print("=== input_2: Date Range ===")
print(f"Range: {fc['date_order'].min().date()} → {fc['date_order'].max().date()}")
print(f"Unique dates: {fc['date_order'].nunique()} (expected 29 for June)")

expected_dates = pd.date_range("2022-06-01", "2022-06-29")
missing_dates = expected_dates.difference(fc["date_order"])
print(f"Missing dates: {list(missing_dates.date) if len(missing_dates) > 0 else 'None'}")

In [ ]:
### input_2: Negative or zero forecasted items
neg_fc = fc[fc["Forecated_items"] <= 0]
print(f"=== input_2: Negative or Zero Forecasted Items ===")
print(f"Rows with Forecated_items <= 0: {len(neg_fc)}")
if len(neg_fc) > 0:
    display(neg_fc)

In [ ]:
### input_2: Day-of-week distribution
print("=== input_2: Day-of-Week Distribution ===")
dow_fc = fc["day_of_week_order"].value_counts().sort_index()
print(dow_fc.to_frame("count"))
print(f"\nDays of week present: {sorted(fc['day_of_week_order'].unique())}")

In [ ]:
### Cross-dataset: day_of_week encoding consistency
print("=== Cross-dataset: Day-of-Week Encoding ===")
print("input_1 day_of_week_order values:", sorted(hist["day_of_week_order"].unique()))
print("input_2 day_of_week_order values:", sorted(fc["day_of_week_order"].unique()))

# Verify encoding is consistent: check a known date against its day_of_week value
sample = hist[["date_order", "day_of_week_order"]].drop_duplicates().head(7)
sample["pandas_dow"] = sample["date_order"].dt.dayofweek + 1  # Monday=1
sample["match"] = sample["day_of_week_order"] == sample["pandas_dow"]
print()
display(sample)

In [ ]:
### Cross-dataset: weekday coverage
print("=== Cross-dataset: Weekday Coverage ===")
hist_dow = set(hist["day_of_week_order"].unique())
fc_dow = set(fc["day_of_week_order"].unique())
missing_in_hist = fc_dow - hist_dow
print(f"Days in forecast not in historical: {missing_in_hist if missing_in_hist else 'None — full coverage'}")

## 3. Data Exploration

In [ ]:
### Lag distribution stats
hist["lag"] = (hist["date_wh_receive"] - hist["date_order"]).dt.days
print("=== input_1: Lag Distribution (days) ===")
print(hist["lag"].describe())
print(f"\nPercentiles:")
for p in [90, 95, 99, 100]:
    print(f"  {p}th: {np.percentile(hist['lag'], p):.0f} days")

**Observations**:
- 50% of all item arrivals happen within 6 days of order.
- 90% of all item arrivals happen within 11 days.
- Single longest observed lag is 26 days.
- The gap between the 99th (16 days) and 100th (26 days) is large — 10 days for just 1% of items. That's a strong signal that the tail beyond 16 days is rare outlier behaviour, not a meaningful pattern worth modelling.
- Note: these percentiles are **row-weighted** — see the item-weighted view in the visualisation below.

In [ ]:
### Lag distribution visualisation (item-weighted)
lag_shares = (
    hist.groupby(["date_order", "lag"])["items"].sum()
    .div(hist.groupby("date_order")["items"].sum(), level="date_order")
    .groupby("lag").mean()
    .reset_index()
    .rename(columns={"items": "avg_share"})
)
lag_shares["cumulative_share"] = lag_shares["avg_share"].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 5))

sns.barplot(data=lag_shares, x="lag", y="avg_share", ax=ax1, color="steelblue", alpha=0.8)
ax1.set_xlabel("Lag (days from order to warehouse receipt)")
ax1.set_ylabel("Average share of items", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

ax2 = ax1.twinx()
ax2.plot(lag_shares["lag"], lag_shares["cumulative_share"], color="darkorange", linewidth=2, marker="o", markersize=4)
ax2.axhline(0.99, color="red", linestyle="--", linewidth=1, label="99% threshold")
ax2.set_ylabel("Cumulative share", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax2.legend(loc="center right")

plt.title("Lag Distribution: Average Item Share per Day (bars) + Cumulative Share (line)")
plt.tight_layout()
plt.show()